In [ ]:
import duckdb
from python_calamine import CalamineWorkbook
from loguru import logger
from pathlib import Path
import polars as pl

In [ ]:
con = duckdb.connect()
con.sql("INSTALL excel;")
con.sql("LOAD excel;")
con.sql("INSTALL sqlite;")
con.sql("LOAD sqlite;")

In [ ]:
file_path: str = r"D:\_Code\_data_raw\xom_data\22th\insurance_complain.xlsx"
data_des_path: str = r"D:\_Code\_lap_and_learn\sql_duckdb\data\xom_data\22th\sqlite_db.db"

In [ ]:
if Path(data_des_path).exists():
    Path(data_des_path).unlink()

In [ ]:
# 1. Đảm bảo thư mục chứa file SQLite phải tồn tại
# (Nếu thư mục output_db chưa có, hàm này sẽ tự tạo ra nó)
Path(data_des_path).parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# get excel sheets
sheet_names: list = CalamineWorkbook.from_path(file_path).sheet_names
sheet_names

In [ ]:
# get schema of each sheet
sheet_schemas: list[dict] = []
for sheet in sheet_names:
    schema = pl.read_excel(file_path, sheet_name=sheet).collect_schema()
    logger.info(f"{sheet = }, {schema = }")
    sheet_schemas.append(sheet_schemas)

In [ ]:
# create new table
con.execute(f"ATTACH IF NOT EXISTS {data_des_path} AS sqlite_db (TYPE SQLITE);")

In [ ]:
# Read data by duckdb -> save to sqlite
for sheet in sheet_names:
  table_name = sheet.strip().lower().replace(" ", "_")
  q = fr"CREATE TABLE IF NOT EXISTS sqlite_db.{table_name} AS SELECT * FROM read_xlsx({file_path}, sheet={sheet}, all_varchar=True)"
  con.sql(q)
  print(f"Complete load excel sheet {sheet} to sqlite table {table_name}")

# con.execute("DETACH sqlite_db;")


In [ ]:
# con.execute("SET sqlite_all_varchar=false;")

In [ ]:
con.sql("SELECT * FROM sqlite_db.brokers LIMIT 1")

In [ ]:
# cast type
# 2. Thực hiện ép kiểu (CAST) dữ liệu an toàn và lưu vào một bảng tạm khác trong bộ nhớ DuckDB

con.execute("""
    CREATE OR REPLACE TEMP TABLE t_brokers AS 
    SELECT 
        TRY_CAST(BrokerID AS INTEGER) AS BrokerID,
        BrokerCode,
        BrokerFullName,
        DistributionNetwork,
        DistributionChannel,
        CommissionScheme
    FROM sqlite_db.brokers;
""")

# 3. Xóa bảng cũ đang bị sai kiểu dữ liệu trong SQLite đi
con.execute("DROP TABLE IF EXISTS sqlite_db.brokers;")

# 4. Ghi đè bảng đã được ép kiểu chuẩn từ bộ nhớ DuckDB ngược trở lại SQLite với TÊN CŨ
con.execute("""
    CREATE TABLE sqlite_db.brokers AS 
    SELECT * FROM t_brokers;
""")


con.sql("SELECT * FROM sqlite_db.brokers")

In [ ]:
con.sql("SELECT * FROM t_brokers")

In [ ]:
import polars as pl

def polars_schema_to_sqlite(schema: pl.Schema) -> dict:
    """Ánh xạ toàn bộ schema của Polars sang kiểu dữ liệu SQLite"""
    mapping = {}
    
    for col_name, pl_type in schema.items():
        # 1. Số nguyên
        if pl_type in (pl.Int8, pl.Int16, pl.Int32, pl.Int64, 
                       pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, 
                       pl.Boolean):
            mapping[col_name] = "INTEGER"
            
        # 2. Số thực
        elif pl_type in (pl.Float32, pl.Float64):
            mapping[col_name] = "REAL"
            
        # 3. Số thập phân chính xác
        elif isinstance(pl_type, pl.Decimal):
            mapping[col_name] = "NUMERIC"
            
        # 4. Chuỗi văn bản & Ngày tháng
        elif pl_type in (pl.String, pl.Categorical, pl.Date, pl.Datetime, pl.Time):
            mapping[col_name] = "TEXT"
            
        # 5. Dữ liệu nhị phân
        elif pl_type == pl.Binary:
            mapping[col_name] = "BLOB"
            
        # 6. Các kiểu phức hợp (List/Struct) -> Lưu dạng Text (JSON)
        else:
            mapping[col_name] = "TEXT"
            
    return mapping

def generate_create_table_sql(table_name: str, schema: pl.Schema) -> str:
    """Tạo chuỗi SQL CREATE TABLE hoàn chỉnh"""
    sqlite_types = polars_schema_to_sqlite(schema)
    column_defs = [f"    {col} {dtype}" for col, dtype in sqlite_types.items()]
    
    sql = f"CREATE TABLE IF NOT EXISTS {table_name} (\n"
    sql += ",\n".join(column_defs)
    sql += "\n);"
    return sql


# Sinh câu lệnh SQL tự động
sql_statement = generate_create_table_sql("sqlite_db.brokers", pl.read_excel(file_path, sheet_name="Brokers").schema)
print(sql_statement)


In [57]:
import os
from pathlib import Path
import duckdb
import polars as pl
import openpyxl

def get_sqlite_type(pl_type) -> str:
    """Ánh xạ kiểu dữ liệu từ Polars sang SQLite Storage Classes"""
    if pl_type in (pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, pl.Boolean):
        return "INTEGER"
    elif pl_type in (pl.Float32, pl.Float64):
        return "REAL"
    elif isinstance(pl_type, pl.Decimal):
        return "NUMERIC"
    elif pl_type in (pl.String, pl.Categorical, pl.Date, pl.Datetime, pl.Time):
        return "TEXT"
    elif pl_type == pl.Binary:
        return "BLOB"
    else:
        return "TEXT"

def process_excel_to_sqlite(excel_path: str, sqlite_path: str):
    # 1. Đảm bảo thư mục chứa SQLite tồn tại
    Path(sqlite_path).parent.mkdir(parents=True, exist_ok=True)

    # 2. Khởi tạo DuckDB và nạp các extension
    con = duckdb.connect()
    con.execute("INSTALL excel; LOAD excel;")
    con.execute("INSTALL sqlite; LOAD sqlite;")

    # Tắt cấu hình all_varchar để DuckDB tuân thủ kiểu dữ liệu của bảng SQLite đã tạo
    con.execute("SET sqlite_all_varchar=false;")

    # Kết nối an toàn tới file SQLite
    con.execute(f"ATTACH IF NOT EXISTS '{sqlite_path}' AS sqlite_db (TYPE SQLITE);")

    # 3. Lấy danh sách tất cả các sheet bằng openpyxl
    wb = openpyxl.load_workbook(excel_path, read_only=True)
    sheet_names = wb.sheetnames

    print(f"🔄 Đang xử lý file: {os.path.basename(excel_path)}")

    for sheet in sheet_names:
        table_name = sheet.strip().lower().replace(" ", "_")

        # 4. Dùng Polars quét schema (sử dụng engine calamine để tối ưu và tránh lỗi syntax tầng ngoài)
        # Nếu chưa cài calamine, bạn có thể chạy: pip install calamine
        try:
            df_schema = pl.read_excel(
                source=excel_path,
                sheet_name=sheet,
                engine="calamine",
                read_options={
                    "has_header": True,
                    "infer_schema_length": 10000,
                    "null_values": ["NULL", "null", "NaN"]
                }
            ).schema
        except Exception:
            # Fallback nếu máy bạn chưa cài engine calamine
            df_schema = pl.read_excel(
                source=excel_path,
                sheet_name=sheet,
                has_header=True
            ).schema

        print(f"Data schema excel: {df_schema}")

        # 5. Xây dựng câu lệnh CREATE TABLE với kiểu dữ liệu tường minh
        column_defs = []
        for col_name, pl_type in df_schema.items():
            sqlite_type = get_sqlite_type(pl_type)
            column_defs.append(f'"{col_name}" {sqlite_type}')
        print(column_defs)
        con.execute(f"DROP TABLE IF EXISTS sqlite_db.{table_name}")
        create_table_sql = f"""
            CREATE TABLE IF NOT EXISTS sqlite_db.{table_name} (
                {", ".join(column_defs)}
            );
        """
        con.execute(create_table_sql)

        # 7. SỬA TẠI ĐÂY: Thêm tham số nullstr để DuckDB tự bỏ qua chữ 'NULL' lỗi khi parse
        insert_sql = f"""
            INSERT INTO sqlite_db.{table_name}
            SELECT * FROM read_xlsx(?, sheet=?, ignore_errors=true);
        """
        con.execute(insert_sql, [excel_path, sheet])
        sqlite_schema = con.sql(f"SHOW sqlite_db.{table_name}")
        print(f"sqlite schema: {sqlite_schema}")
        print(f"   └── Đã nạp xong sheet '{sheet}' vào bảng sqlite_db.{table_name}")

    # 8. Kết thúc quy trình, ngắt kết nối an toàn
    con.execute("DETACH sqlite_db;")
    con.close()
    print(f"🎉 Hoàn thành xử lý file Excel vào {sqlite_path}\n")

if __name__ == "__main__":
    # Đảm bảo bạn đã định nghĩa biến file_path và data_des_path trước khi gọi hàm
    file_path = "/home/user/py-pannguyen-learn/data/raw/21th/Used_Car_Sales_dataset.xlsx"
    data_des_path = "/home/user/py-pannguyen-learn/data/staging/21th/user_car_sale.db"
    process_excel_to_sqlite(file_path, data_des_path)


🔄 Đang xử lý file: Used_Car_Sales_dataset.xlsx
Data schema excel: Schema([('dateCrawled', Datetime(time_unit='ms', time_zone=None)), ('name', String), ('seller', String), ('offerType', String), ('price', Int64), ('abtest', String), ('vehicleType', String), ('yearOfRegistration', Int64), ('gearbox', String), ('powerPS', Int64), ('model', String), ('kilometer', Int64), ('monthOfRegistration', Int64), ('fuelType', String), ('brand', String), ('notRepairedDamage', String), ('dateCreated', Date), ('nrOfPictures', Int64), ('postalCode', Int64), ('Lattitude', Float64), ('Longitude', Float64), ('lastSeen', Datetime(time_unit='ms', time_zone=None))])
['"dateCrawled" TEXT', '"name" TEXT', '"seller" TEXT', '"offerType" TEXT', '"price" INTEGER', '"abtest" TEXT', '"vehicleType" TEXT', '"yearOfRegistration" INTEGER', '"gearbox" TEXT', '"powerPS" INTEGER', '"model" TEXT', '"kilometer" INTEGER', '"monthOfRegistration" INTEGER', '"fuelType" TEXT', '"brand" TEXT', '"notRepairedDamage" TEXT', '"dateCreat

In [1]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL sqlite; LOAD sqlite;")
data_des_path = "/home/user/py-pannguyen-learn/data/staging/21th/user_car_sale.db"
con.execute(f"ATTACH IF NOT EXISTS '{data_des_path}' AS sqlite_db (TYPE SQLITE);")
con.sql("SHOW ALL TABLES;")

databaseVARCHAR,schemaVARCHAR,nameVARCHAR,column_namesVARCHAR[],column_typesVARCHAR[],temporaryBOOLEAN
sqlite_db,main,sheet1,"['dateCrawled', 'name', 'seller', 'offerType', 'price', 'abtest', 'vehicleType', 'yearOfRegistration', 'gearbox', 'powerPS', 'model', 'kilometer', 'monthOfRegistration', 'fuelType', 'brand', 'notRepairedDamage', 'dateCreated', 'nrOfPictures', 'postalCode', 'Lattitude', 'Longitude', 'lastSeen']","['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'BIGINT', 'VARCHAR', 'VARCHAR', 'BIGINT', 'VARCHAR', 'BIGINT', 'VARCHAR', 'BIGINT', 'BIGINT', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'BIGINT', 'BIGINT', 'DOUBLE', 'DOUBLE', 'VARCHAR']",False


In [4]:
con.sql("SELECT COUNT(*) FROM sqlite_db.sheet1")

count_star()BIGINT
371404


In [2]:
con.sql("DESCRIBE sqlite_db.sheet1")

column_nameVARCHAR,column_typeVARCHAR,nullVARCHAR,keyVARCHAR,defaultVARCHAR,extraVARCHAR
dateCrawled,VARCHAR,YES,NULL,NULL,NULL
name,VARCHAR,YES,NULL,NULL,NULL
seller,VARCHAR,YES,NULL,NULL,NULL
offerType,VARCHAR,YES,NULL,NULL,NULL
price,BIGINT,YES,NULL,NULL,NULL
abtest,VARCHAR,YES,NULL,NULL,NULL
vehicleType,VARCHAR,YES,NULL,NULL,NULL
yearOfRegistration,BIGINT,YES,NULL,NULL,NULL
gearbox,VARCHAR,YES,NULL,NULL,NULL
powerPS,BIGINT,YES,NULL,NULL,NULL


In [26]:
con.sql("SHOW sqlite_db.sheet1")

column_nameVARCHAR,column_typeVARCHAR,nullVARCHAR,keyVARCHAR,defaultVARCHAR,extraVARCHAR
dateCrawled,VARCHAR,YES,NULL,NULL,NULL
name,VARCHAR,YES,NULL,NULL,NULL
seller,VARCHAR,YES,NULL,NULL,NULL
offerType,VARCHAR,YES,NULL,NULL,NULL
price,BIGINT,YES,NULL,NULL,NULL
abtest,VARCHAR,YES,NULL,NULL,NULL
vehicleType,VARCHAR,YES,NULL,NULL,NULL
yearOfRegistration,BIGINT,YES,NULL,NULL,NULL
gearbox,VARCHAR,YES,NULL,NULL,NULL
powerPS,BIGINT,YES,NULL,NULL,NULL


In [ ]:
con.sql("""
        SELECT SUM(CASE dateCrawled WHEN NULL THEN 1 ELSE 0 END) +
       SUM(CASE name WHEN NULL THEN 1 ELSE 0 END) +
       SUM(CASE seller WHEN NULL THEN 1 ELSE 0 END) 
    FROM sqlite_db.sheet1;
        
        """)

In [ ]:
con.sql("""
        SELECT 
        SUM(CASE dateCrawled WHEN NULL THEN 1 ELSE 0 END) AS dateCrawled_null,
        SUM(CASE name WHEN NULL THEN 1 ELSE 0 END) AS name_null,
        SUM(CASE seller WHEN NULL THEN 1 ELSE 0 END) AS seller_null,
        SUM(CASE offerType WHEN NULL THEN 1 ELSE 0 END) AS offerType_null,
        SUM(CASE price WHEN NULL THEN 1 ELSE 0 END) AS price_null,
        SUM(CASE abtest WHEN NULL THEN 1 ELSE 0 END) AS abtest_null,
        SUM(CASE vehicleType WHEN NULL THEN 1 ELSE 0 END) AS vehicleType_null,
        SUM(CASE yearOfRegistration WHEN NULL THEN 1 ELSE 0 END) AS yearOfRegistration_null,
        SUM(CASE gearbox WHEN NULL THEN 1 ELSE 0 END) AS gearbox_null,
        SUM(CASE powerPS WHEN NULL THEN 1 ELSE 0 END) AS powerPS_null,
    FROM sqlite_db.sheet1;
        
        """)

In [ ]:
con.sql("SELECT COUNT(DISTINCT(name)) FROM sqlite_db.sheet1")

In [ ]:
con.sql("""
        
        SELECT 
        COUNT(DISTINCT(dateCrawled)) AS distinct_dateCrawled,
        COUNT(DISTINCT(name)) AS distinct_name,
        COUNT(DISTINCT(seller)) AS distinct_seller,
        COUNT(DISTINCT(offerType)) AS distinct_offerType,
        COUNT(DISTINCT(price)) AS distinct_price,
        COUNT(DISTINCT(abtest)) AS distinct_abtest,
        COUNT(DISTINCT(vehicleType)) AS distinct_vehicleType,
        COUNT(DISTINCT(yearOfRegistration)) AS distinct_yearOfRegistration,
        COUNT(DISTINCT(gearbox)) AS distinct_gearbox,
        COUNT(DISTINCT(powerPS)) AS distinct_powerPS,
        FROM sqlite_db.sheet1
""")

In [ ]:
con.sql("""
        SELECT
            strptime(dateCrawled, %Y-%m-%d %H:%M:%S) AS dateCrawled
        FROM sqlite_db.sheet1
        
        """)

dateCrawledTIMESTAMP
2016-03-19 16:52:13
2016-03-18 14:38:13
2016-03-21 18:58:48
2016-03-23 23:52:06
2016-04-05 10:49:28
2016-03-30 18:50:42
2016-03-24 15:57:57
2016-03-26 17:50:33
2016-03-15 13:56:10
2016-03-15 08:56:01


In [ ]:
# # 1. Create the new table directly in the SQLite database with dateCrawled cast to TIMESTAMP
# con.sql("""
#     CREATE TABLE sqlite_db.sheet1_new AS
#     SELECT
#         TRY_CAST(dateCrawled AS TIMESTAMP) AS dateCrawled,
#         name,
#         seller,
#         offerType,
#         price,
#         abtest,
#         vehicleType,
#         yearOfRegistration,
#         gearbox,
#         powerPS,
#         model,
#         kilometer,
#         monthOfRegistration,
#         fuelType,
#         brand,
#         notRepairedDamage,
#         dateCreated,
#         nrOfPictures,
#         postalCode,
#         Lattitude,
#         Longitude,
#         lastSeen
#     FROM sqlite_db.sheet1;
# """)

# # 2. Drop the original table in SQLite
# con.sql("""DROP TABLE sqlite_db.sheet1;""")

# # 3. Rename the new table to the original name within SQLite
# con.sql("""ALTER TABLE sqlite_db.sheet1_new RENAME TO sheet1;""")

# # 4. Verify the updated schema
# con.sql("DESCRIBE sqlite_db.sheet1")

In [5]:
con.sql("SELECT TRY_CAST(dateCrawled AS TIMESTAMP) AS dateCrawled FROM sqlite_db.sheet1 ORDER BY dateCrawled ASC")

dateCrawledTIMESTAMP
2016-03-05 14:06:22
2016-03-05 14:06:23
2016-03-05 14:06:24
2016-03-05 14:06:24
2016-03-05 14:06:24
2016-03-05 14:06:25
2016-03-05 14:06:25
2016-03-05 14:06:26
2016-03-05 14:06:27
2016-03-05 14:06:27


In [11]:
con.sql("""
        SELECT 
        MIN(TRY_CAST(dateCrawled AS TIMESTAMP)) AS min_dateCrawled,
        MAX(TRY_CAST(dateCrawled AS TIMESTAMP)) AS max_dateCrawled
        FROM sqlite_db.sheet1
""")

min_dateCrawledTIMESTAMP,max_dateCrawledTIMESTAMP
2016-03-05 14:06:22,2016-04-07 14:36:58


exercise: /home/user/py-pannguyen-learn/data/raw/21th/Đề bài.docx


In [17]:
con.sql("SELECT COUNT(DISTINCT(name)) FROM sqlite_db.sheet1")

"count(DISTINCT ""name"")BIGINT"
224517


In [39]:
con.sql("""
    WITH count AS (
        SELECT
            name,
            COUNT(*) OVER (PARTITION BY name) AS num_of_duplicate_name
        FROM sqlite_db.sheet1
    )
    SELECT DISTINCT name, num_of_duplicate_name
    FROM count
    WHERE num_of_duplicate_name > 1
    ORDER BY num_of_duplicate_name DESC, name DESC
""")

nameVARCHAR,num_of_duplicate_nameBIGINT
Opel_Corsa,812
Ford_Fiesta,776
BMW_318i,629
Volkswagen_Golf_1.4,607
Renault_Twingo,582
Opel_Corsa_b,531
BMW_316i,525
Volkswagen_Polo,494
BMW_320i,493
Opel_astra,457


In [44]:
con.sql("""
    WITH count AS (
        SELECT
            name,
            COUNT(*) OVER (PARTITION BY name) AS num_of_duplicate_name
        FROM sqlite_db.sheet1
    ),
    min_max_count AS (
        SELECT
            MIN(num_of_duplicate_name) AS min_count,
            MAX(num_of_duplicate_name) AS max_count
        FROM count
    )
    SELECT
        DISTINCT name,
        num_of_duplicate_name
    FROM count
    WHERE num_of_duplicate_name == (SELECT max_count FROM min_max_count)
    ORDER BY num_of_duplicate_name DESC
""")

nameVARCHAR,num_of_duplicate_nameBIGINT
Opel_Corsa,812


In [ ]:
# Tổng quy mô thị trường xe cũ trong tệp dữ liệu là bao nhiêu xe?
con.sql("SELECT COUNT(*) FROM sqlite_db.sheet1")

count_star()BIGINT
371404


In [ ]:
# Mức giá bán trung bình (Average Price) và số kilomet trung bình của thị trường là bao nhiêu?
con.sql("""
        SELECT
            ROUND(AVG(price), 2) AS avg_price,
            ROUND(AVG(kilometer), 2) AS avg_kilometer
        FROM sqlite_db.sheet1
        """)

avg_priceDOUBLE,avg_kilometerDOUBLE
17300.85,125618.96


In [3]:
import polars as pl
pl.Config(set_tbl_rows=-1, set_tbl_cols=-1)

In [68]:
con.sql("SELECT DISTINCT(vehicleType) FROM sqlite_db.sheet1")

vehicleTypeVARCHAR
NULL
andere
kombi
suv
bus
limousine
cabrio
coupe
kleinwagen


In [ ]:
con.sql("""
    SELECT COUNT(*)
    FROM sqlite_db.sheet1
    WHERE vehicleType IS NULL
        """)

count_star()BIGINT
37854


In [ ]:
con.sql("""
        UPDATE sqlite_db.sheet1
        SET vehicleType = 'unknown'
        WHERE vehicleType IS NULL
        """)

In [ ]:
con.sql("SELECT DISTINCT(vehicleType) FROM sqlite_db.sheet1")

In [ ]:
con.sql("""
    SELECT COUNT(*)
    FROM sqlite_db.sheet1
    WHERE vehicleType IS NULL
        """)

In [9]:
con.sql("""SELECT DISTINCT yearOfRegistration 
        FROM sqlite_db.sheet1 
        WHERE yearOfRegistration < 1910 OR yearOfRegistration > 2026
        ORDER BY yearOfRegistration ASC
        LIMIT 100
""")

yearOfRegistrationBIGINT
1000
1001
1039
1111
1200
1234
1253
1255
1300
1400


In [10]:
con.sql("""SELECT DISTINCT monthOfRegistration 
        FROM sqlite_db.sheet1 
        WHERE monthOfRegistration < 1 OR monthOfRegistration > 12
        ORDER BY monthOfRegistration ASC
        LIMIT 100
""").pl()

monthOfRegistration
i64
0


In [11]:
con.sql("""SELECT DISTINCT price 
        FROM sqlite_db.sheet1 
        WHERE price <= 0
        ORDER BY price ASC
        LIMIT 100
""").pl()

price
i64
0


In [17]:
con.sql("""SELECT DISTINCT TRY_CAST(kilometer AS BIGINT) as kilometer
        FROM sqlite_db.sheet1 
        WHERE kilometer <= 0
        ORDER BY kilometer ASC
        LIMIT 100
""")

kilometerBIGINT


In [25]:
con.sql("""SELECT 
        DISTINCT strftime(TRY_CAST(dateCreated AS TIMESTAMP), '%Y-%m-%d %H:%M:%S') as dateCreated
        FROM sqlite_db.sheet1 
        WHERE TRY_CAST(dateCreated AS TIMESTAMP) >= '2026-11-06 00:00:00' OR TRY_CAST(dateCreated AS TIMESTAMP) <= '1900-01-01 00:00:00'
        ORDER BY TRY_CAST(dateCreated AS TIMESTAMP) ASC
        LIMIT 100
""")

dateCreatedVARCHAR


In [26]:
con.sql("""SELECT 
        DISTINCT strftime(TRY_CAST(lastSeen AS TIMESTAMP), '%Y-%m-%d %H:%M:%S') as lastSeen
        FROM sqlite_db.sheet1 
        WHERE TRY_CAST(lastSeen AS TIMESTAMP) >= '2026-11-06 00:00:00' OR TRY_CAST(lastSeen AS TIMESTAMP) <= '1900-01-01 00:00:00'
        ORDER BY TRY_CAST(lastSeen AS TIMESTAMP) ASC
        LIMIT 100
""")

lastSeenVARCHAR


In [33]:
con.sql("EXPLAIN SELECT * FROM sqlite_db.sheet1" )

explain_keyVARCHAR,explain_valueVARCHAR
physical_plan,"╭─ Sqlite Scan ────────────────────────────────────────────────────╮ │ Table: sheet1 │ │ File: │ │ /home/user/py-pannguyen-learn/data/staging/21th/user_car_sale.db │ │ Projections: dateCrawled, name, seller, offerType, price, │ │ abtest, vehicleType, yearOfRegistration, gearbox, │ │ powerPS, model, kilometer, monthOfRegistration, │ │ fuelType, brand, notRepairedDamage, dateCreated, │ │ nrOfPictures, postalCode, Lattitude, Longitude, │ │ lastSeen │ │ ~371,403 rows │ ╰──────────────────────────────────────────────────────────────────╯"


In [52]:
"""
Mức độ mất giá của xe diễn ra như thế nào theo thời gian? 
Hãy tính toán giá trung bình của xe theo từng năm đăng ký (yearOfRegistration).
"""
con.sql("""
        WITH valid_data AS (
            SELECT 
                vehicleType,
                yearOfRegistration,
                price,
            FROM sqlite_db.sheet1
            WHERE vehicleType IS NOT NULL AND 
            yearOfRegistration IS NOT NULL AND 
            price IS NOT NULL
            AND (yearOfRegistration >= 1900 AND yearOfRegistration <= 2026)
            AND price > 0
        )
        PIVOT valid_data 
            ON yearOfRegistration
            USING ROUND(AVG(price),2) AS avg_price
            GROUP BY vehicleType
            ORDER BY vehicleType ASC
        """)


vehicleTypeVARCHAR,1910_avg_priceDOUBLE,1911_avg_priceDOUBLE,1923_avg_priceDOUBLE,1925_avg_priceDOUBLE,1927_avg_priceDOUBLE,1928_avg_priceDOUBLE,1929_avg_priceDOUBLE,1930_avg_priceDOUBLE,1931_avg_priceDOUBLE,1932_avg_priceDOUBLE,1933_avg_priceDOUBLE,1934_avg_priceDOUBLE,1935_avg_priceDOUBLE,1936_avg_priceDOUBLE,1937_avg_priceDOUBLE,1938_avg_priceDOUBLE,1939_avg_priceDOUBLE,1940_avg_priceDOUBLE,1941_avg_priceDOUBLE,1942_avg_priceDOUBLE,1943_avg_priceDOUBLE,1944_avg_priceDOUBLE,1945_avg_priceDOUBLE,1946_avg_priceDOUBLE,1947_avg_priceDOUBLE,1948_avg_priceDOUBLE,1949_avg_priceDOUBLE,1950_avg_priceDOUBLE,1951_avg_priceDOUBLE,1952_avg_priceDOUBLE,1953_avg_priceDOUBLE,1954_avg_priceDOUBLE,1955_avg_priceDOUBLE,1956_avg_priceDOUBLE,1957_avg_priceDOUBLE,1958_avg_priceDOUBLE,1959_avg_priceDOUBLE,1960_avg_priceDOUBLE,1961_avg_priceDOUBLE,1962_avg_priceDOUBLE,1963_avg_priceDOUBLE,1964_avg_priceDOUBLE,1965_avg_priceDOUBLE,1966_avg_priceDOUBLE,1967_avg_priceDOUBLE,1968_avg_priceDOUBLE,1969_avg_priceDOUBLE,1970_avg_priceDOUBLE,1971_avg_priceDOUBLE,1972_avg_priceDOUBLE,1973_avg_priceDOUBLE,1974_avg_priceDOUBLE,1975_avg_priceDOUBLE,1976_avg_priceDOUBLE,1977_avg_priceDOUBLE,1978_avg_priceDOUBLE,1979_avg_priceDOUBLE,1980_avg_priceDOUBLE,1981_avg_priceDOUBLE,1982_avg_priceDOUBLE,1983_avg_priceDOUBLE,1984_avg_priceDOUBLE,1985_avg_priceDOUBLE,1986_avg_priceDOUBLE,1987_avg_priceDOUBLE,1988_avg_priceDOUBLE,1989_avg_priceDOUBLE,1990_avg_priceDOUBLE,1991_avg_priceDOUBLE,1992_avg_priceDOUBLE,1993_avg_priceDOUBLE,1994_avg_priceDOUBLE,1995_avg_priceDOUBLE,1996_avg_priceDOUBLE,1997_avg_priceDOUBLE,1998_avg_priceDOUBLE,1999_avg_priceDOUBLE,2000_avg_priceDOUBLE,2001_avg_priceDOUBLE,2002_avg_priceDOUBLE,2003_avg_priceDOUBLE,2004_avg_priceDOUBLE,2005_avg_priceDOUBLE,2006_avg_priceDOUBLE,2007_avg_priceDOUBLE,2008_avg_priceDOUBLE,2009_avg_priceDOUBLE,2010_avg_priceDOUBLE,2011_avg_priceDOUBLE,2012_avg_priceDOUBLE,2013_avg_priceDOUBLE,2014_avg_priceDOUBLE,2015_avg_priceDOUBLE,2016_avg_priceDOUBLE,2017_avg_priceDOUBLE,2018_avg_priceDOUBLE
andere,150.33,NULL,NULL,NULL,NULL,NULL,1.0,12400.0,4700.0,NULL,NULL,2117.0,16000.0,NULL,9500.0,4500.0,NULL,NULL,26900.0,NULL,NULL,NULL,12000.0,1500.0,26500.0,NULL,NULL,5200.0,16583.0,NULL,2790777.75,17833.0,17300.0,1300.0,3500.0,11100.0,8772.25,257237.25,3297.5,10700.0,12474.75,7741.5,8500.0,9723.2,3675.0,9700.0,5890.14,6812.0,9602.75,8050.0,7471.29,11119644.33,5781.13,8685.0,9983.33,7100.0,8954.93,8117.65,7957.14,6334.62,4529.0,4995.93,7214.31,2593.08,2643.78,4954.86,2029.96,3635.47,2588.38,2351.12,2245.85,1353.14,1814.1,14811716.84,1413.26,1884.87,1701.11,2080.63,2756.01,3543.76,3023.47,3886.62,4443.34,4914.24,6297.96,7307.18,8058.41,7741.65,9972.66,10999.31,14076.92,15893.81,19052.15,47897.62,3450.0,NULL
bus,750.5,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,6666.0,NULL,500.0,5500.0,NULL,41250.0,57599.0,15697.5,32700.0,21939.67,15613.75,19900.0,23266.67,30857.83,47739.8,11450.0,11595.0,10486.44,17961.75,11743.75,14518.55,15396.0,11723.8,10167.93,15477.13,9973.27,6646.35,6411.19,105439.8,5645.13,8689.27,4602.92,5837.61,5485.72,5854.67,4583.81,4680.44,4803.94,2832.37,3763.29,2957.88,3331.82,2834.74,2918.83,2249.02,2432.6,2363.87,2526.45,2843.93,2865.47,3333.44,4250.47,4874.59,42602.42,11857.07,8279.9,9063.85,12061.29,14062.82,15872.27,17403.07,22761.16,28361.16,28871.83,6175.0,NULL
cabrio,1.0,30000.0,20833.33,1800.0,16500.0,9999.0,15300.0,26500.0,27500.0,23633.33,235000.0,275000.0,23495.0,25950.0,30250.0,8500.0,26600.0,NULL,NULL,NULL,41900.0,17000.0,NULL,NULL,44475.0,NULL,NULL,35000.0,67475.0,205000.0,42899.75,59500.0,105766.5,83800.0,39040.0,121500.0,65715.0,68079.21,47790.0,29728.43,39799.93,51483.17,24379.47,33690.07,80165.17,22311.31,33726.0,47707.74,20818.81,40158.07,21934.75,13490.57,15878.86,12149.97,13374.85,13541.59,12908.07,12501.62,11631.21,16956.23,13950.39,13835.81,14620.14,15015.55,12561.07,10185.21,10909.89,9549.76,80

In [50]:
con.sql("""
        WITH valid_data AS (
            SELECT *
            FROM sqlite_db.sheet1
            WHERE vehicleType IS NOT NULL AND 
            yearOfRegistration IS NOT NULL AND 
            price IS NOT NULL
            AND (yearOfRegistration >= 1900 AND yearOfRegistration <= 2026)
            AND price > 0
        )
    SELECT
        vehicleType,
        yearOfRegistration,
        ROUND(AVG(price), 2) as avg_price,
    FROM valid_data
    GROUP BY vehicleType, yearOfRegistration
    ORDER BY vehicleType ASC, yearOfRegistration ASC
        """).pl().sort(by="yearOfRegistration", descending=False).pivot(on="yearOfRegistration", index="vehicleType", values="avg_price").sort(by="vehicleType", descending=False)

vehicleType,1910,1911,1923,1925,1927,1928,1929,1930,1931,1932,1933,1934,1935,1936,1937,1938,1939,1940,1941,1942,1943,1944,1945,1946,1947,1948,1949,1950,1951,1952,1953,1954,1955,1956,1957,1958,1959,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""andere""",150.33,null,null,null,null,null,1.0,12400.0,4700.0,null,null,2117.0,16000.0,null,9500.0,4500.0,null,null,26900.0,null,null,null,12000.0,1500.0,26500.0,null,null,5200.0,16583.0,null,2.7908e6,17833.0,17300.0,1300.0,3500.0,11100.0,8772.25,257237.25,3297.5,10700.0,12474.75,7741.5,8500.0,9723.2,3675.0,9700.0,5890.14,6812.0,9602.75,8050.0,7471.29,1.1120e7,5781.13,8685.0,9983.33,7100.0,8954.93,8117.65,7957.14,6334.62,4529.0,4995.93,7214.31,2593.08,2643.78,4954.86,2029.96,3635.47,2588.38,2351.12,2245.85,1353.14,1814.1,1.4812e7,1413.26,1884.87,1701.11,2080.63,2756.01,3543.76,3023.47,3886.62,4443.34,4914.24,6297.96,7307.18,8058.41,7741.65,9972.66,10999.31,14076.92,15893.81,19052.15,47897.62,3450.0,null
"""bus""",750.5,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,6666.0,null,500.0,5500.0,null,41250.0,57599.0,15697.5,32700.0,21939.67,15613.75,19900.0,23266.67,30857.83,47739.8,11450.0,11595.0,10486.44,17961.75,11743.75,14518.55,15396.0,11723.8,10167.93,15477.13,9973.27,6646.35,6411.19,105439.8,5645.13,8689.27,4602.92,5837.61,5485.72,5854.67,4583.81,4680.44,4803.94,2832.37,3763.29,2957.88,3331.82,2834.74,2918.83,2249.02,2432.6,2363.87,2526.45,2843.93,2865.47,3333.44,4250.47,4874.59,42602.42,11857.07,8279.9,9063.85,12061.29,14062.82,15872.27,17403.07,22761.16,28361.16,28871.83,6175.0,null
"""cabrio""",1.0,30000.0,20833.33,1800.0,16500.0,9999.0,15300.0,26500.0,27500.0,23633.33,235000.0,275000.0,23495.0,25950.0,30250.0,8500.0,26600.0,null,null,null,41900.0,17000.0,null,null,44475.0,null,null,35000.0,67475.0,205000.0,42899.75,59500.0,105766.5,83800.0,39040.0,121500.0,65715.0,68079.21,47790.0,29728.43,39799.93,51483.17,24379.47,33690.07,80165.17,22311.31,33726.0,47707.74,20818.81,40158.07,21934.75,13490.57,15878.86,12149.97,13374.85,13541.59,12908.07,12501.62,11631.21,16956.23,13950.39,13835.81,14620.14,15015.55,12561.07,10185.21,10909.89,9549.76,8062.47,6399.3,6066.13,4082.56,4651.21,137771.96,3635.97,4161.85,4552.1,5417.64,6923.38,6577.74,7339.39,7842.9,8834.99,10215.96,11919.99,14057.11,16704.56,18989.98,21578.42,23504.36,27504.05,30124.06,37234.74,101129.52,4350.0,null
"""coupe""",null,null,null,null,55555.0,null,20000.0,null,39999.0,null,null,55000.0,null,null,10950.0,null,58500.0,null,null,null,null,null,null,27600.0,17800.0,17399.5,null,9400.33,28900.0,null,null,null,95000.0,11920.0,47325.0,27728.57,30275.0,1688816.5,12100.0,13999.5,37783.33,13555.5,24595.83,19591.8,257905.33,33093.37,33318.83,16987.35,42448.04,18510.78,18955.82,22138.81,12434.25,23474.82,18585.46,11433.51,13058.63,30065.73,24904.76,9880.1,10172.73,18804.83,20505.59,19113.92,13829.53,10467.95,7232.24,9132.79,7974.57,269461.45,6072.85,3826.2,4669.46,3724.98,4083.73,4578.24,3395.89,3995.89,4675.71,5384.72,21881.3,10104.31,19356.32,23614.74,15451.62,31909.12,117276.17,22495.29,24897.9,26368.98,28309.71,36906.37,50566.21,248638.27,1300.0,1650.0
"""kleinwagen""",863.17,null,null,null,null,null,null,null,null,null,17200.0,null,null,1000.0,4150.0,10249.5,26500.0,15000.0,null,null,null,null,500.0,null,null